# 3.3 · Linear-time sorts

*Sorting* · 4 exercises

Beat the N log N bound for comparison sorts by not comparing: count small keys, sort digit by digit, or spread values over buckets.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-3.3) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-3.3) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-3.3)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 3.3.1 · Counting sort  `Core`

**Task.** Every value in `nums` is an integer from `0` to `max_value`. Write `counting_sort(nums, max_value)` that returns a new sorted list by counting how often each value occurs, with no comparisons between values.

**Example.** `[3, 0, 2, 3, 1], 3` -> `[0, 1, 2, 3, 3]`

**Big O.** O(N + K) time and O(K) space for keys 0..K: one pass to count, one pass over the counts to write the output.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-3-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-3-1)

In [ ]:
# 3.3.1 Counting sort
def counting_sort(nums, max_value):
    raise NotImplementedError

In [ ]:
def test_ex_3_3_1():
    assert counting_sort([3, 0, 2, 3, 1], 3) == [0, 1, 2, 3, 3]
    assert counting_sort([], 5) == []
    assert counting_sort([4, 4, 4], 4) == [4, 4, 4]
    assert counting_sort([9, 0, 5, 0], 9) == [0, 0, 5, 9]


test_ex_3_3_1()
print('3.3.1 passed')

### 3.3.2 · Sort to a given key order  `Core`

**Task.** `order` lists a few distinct keys, and every value in `values` is one of them. Write `sort_by_order(values, order)` that returns the values grouped in the order of `order`, using one counting pass and a dictionary of counts.

**Example.** `[2, 0, 1, 2, 0], [2, 0, 1]` -> `[2, 2, 0, 0, 1]`

**Big O.** O(N + K) time, O(K) space for K keys: count each key once, then write the keys out in the order asked for.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-3-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-3-2)

In [ ]:
# 3.3.2 Sort to a given key order
def sort_by_order(values, order):
    raise NotImplementedError

In [ ]:
def test_ex_3_3_2():
    assert sort_by_order([2, 0, 1, 2, 0], [2, 0, 1]) == [2, 2, 0, 0, 1]
    assert sort_by_order([], [1, 2]) == []
    assert sort_by_order([5, 5, 7], [7, 5, 9]) == [7, 5, 5]
    assert sort_by_order([3, 1, 3], [1, 3]) == [1, 3, 3]


test_ex_3_3_2()
print('3.3.2 passed')

### 3.3.3 · Radix sort  `Practice`

**Task.** Write `radix_sort(nums)` that sorts non-negative integers and returns a new list, digit by digit from the lowest: each pass distributes the values into ten buckets by the current digit, keeping their order, and reads the buckets back in order.

**Example.** `[329, 57, 640, 4, 1000, 72]` -> `[4, 57, 72, 329, 640, 1000]`

**Big O.** O(D * (N + 10)) time for D-digit keys, O(N) space: each digit pass is a stable distribution, so earlier digits stay in order within equal later ones.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-3-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-3-3)

In [ ]:
# 3.3.3 Radix sort
def radix_sort(nums):
    raise NotImplementedError

In [ ]:
def test_ex_3_3_3():
    assert radix_sort([329, 57, 640, 4, 1000, 72]) == [4, 57, 72, 329, 640, 1000]
    assert radix_sort([]) == []
    assert radix_sort([0, 0, 1]) == [0, 0, 1]
    assert radix_sort([21, 12, 11, 22]) == [11, 12, 21, 22]


test_ex_3_3_3()
print('3.3.3 passed')

### 3.3.4 · Bucket sort  `Practice`

**Task.** Every value in `values` is a float in `[0, 1)`. Write `bucket_sort(values)` that returns a new sorted list: put each value `v` into bucket `int(v * n)` of `n = len(values)` buckets, sort each (small) bucket, and join them.

**Example.** `[0.42, 0.07, 0.93, 0.5, 0.31]` -> `[0.07, 0.31, 0.42, 0.5, 0.93]`

**Big O.** O(N) expected time for evenly spread values, O(N) space: each of the N buckets gets about one value, so sorting the buckets is cheap; the worst case is one full bucket.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-3-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-3-4)

In [ ]:
# 3.3.4 Bucket sort
def bucket_sort(values):
    raise NotImplementedError

In [ ]:
def test_ex_3_3_4():
    assert bucket_sort([0.42, 0.07, 0.93, 0.5, 0.31]) == [0.07, 0.31, 0.42, 0.5, 0.93]
    assert bucket_sort([]) == []
    assert bucket_sort([0.25, 0.2, 0.21, 0.0]) == [0.0, 0.2, 0.21, 0.25]
    assert bucket_sort([0.99]) == [0.99]


test_ex_3_3_4()
print('3.3.4 passed')